# 02 - LLaMEA Evolutionary Synthesis Pipeline

Multi-process parallel algorithm discovery across the BBOB problem benchmark matrix with SQLite persistence and auto-resumption.

The current configuration selects **f1, f8, f11, f15 and f21**, dimensions **2, 10 and 50**, and noise levels **0, 0.05, 0.1 and 0.2**, with all four prompt strategies in Explicit and Implicit modes. Each candidate receives **100,000 objective calls** and the existing **30-second timeout**. All candidates in a fresh synthesis session use **instance 1**; resumed sessions retain their recorded instance.


**Before running a new study:** complete the manual storage cutover to fresh database/code/checkpoint/result directories and restart the kernel. Editing configurations does not activate the separately prepared staging area, and the current completion checks can otherwise reuse experiments from the old database. The old study is preserved in the verified archive.


In [22]:
import os
import sys
from pathlib import Path

# Ensure project root & src are in path
cwd = Path(".").resolve()
root_dir = cwd.parent if cwd.name == "notebooks" else cwd
os.chdir(root_dir)
src_dir = root_dir / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

# Keep module identities stable for multiprocessing/pickling.
# After updating project code or configuration, restart the kernel instead of evicting modules.

from evolution.infra.di import EvolutionProvider
from evolution.application.campaign.audit import CampaignAudit
from evolution.application.campaign.service import SynthesisCampaign
from shared.infra.lifecycle import WorkflowScope

print("[OK] LLaMEA Evolution Pipeline initialized.")


[OK] LLaMEA Evolution Pipeline initialized.


## 1. LLM Provider Connection
Initialize and verify the LLM connection configured in your `.env` file.


In [23]:
# Prepare services once; infrastructure owns shutdown cleanup.
if "services" in globals():
    services.close()
services = WorkflowScope(
    EvolutionProvider(test_generation_on_startup=True), context={}, request_context={}
)
print("Checking model access and one short generation request...", flush=True)
campaign = services.get(SynthesisCampaign)
audit = services.get(CampaignAudit)
print("=" * 65)
print(f"Model Target: {campaign.model_name}")
print("=" * 65)

Diagnostic log: /Users/nicolaibrahim/Desktop/proj/AAD_LLM/logs/evolution/2026-10-29_21-09-00_UTC-48b62684/diagnostics.log
Checking model access and one short generation request...
Model Target: gwdg.qwen3-coder-next


## 2. Load Configuration & Experiment Matrix
The setup cell defines explicit target problems, dimensions, prompts, noise levels and execution parameters. The audit checks current completion status in SQLite. Setup does not run migrations or launch synthesis. Databases without the optional `synthesis_seed` column remain supported: seeds are not persisted there, but problem/noise seed assignment is unchanged.

The audit and run calls contain explicit study parameters; optional experiment selections belong to the run call. Pass dynamic requests to `run(...)` in the execution cell: `recover=True` discovers interrupted runs and unresolved synthesis failures for the active model only where no completed valid champion exists. `rerun_experiment_ids=[...]` replaces those exact experiment records from generation one, retaining their IDs and deleting old iterations, diagnostics, champion state, code and checkpoints. Stop any other campaign using these IDs first. Automatic recovery resumes one pending record or replaces one failed record without creating new experiment rows; any completed valid champion satisfies automatic recovery, regardless of failed candidate iterations, other failed sessions, or leftover queued records. `resume_experiment_ids=[...]` only resumes those running records and cannot be combined with recovery or replacement. Calling `run(**synthesis_parameters)` without selections uses normal full-matrix scheduling. Each call is independent; omit manual IDs on later launches after successful repairs. IDs belong to the current database; review them after an import. The audit counts conditions with a valid completed champion; running and failed record counts are history, not reasons to rerun a covered condition. The standard protocol uses 10 candidate iterations in one synthesis session, not 10 independent trials. A valid finite synthesis result may still perform poorly in benchmarking; request a manual rerun to improve it. No migrations are run.

In [24]:
matrix_df_result = audit.inspect(
    problem_targets=[{"id": problem, "dimensions": [2, 10, 50]} for problem in [1, 8, 11, 15, 21]],
    noise_conditions=[
        {"std": 0.0, "noise_model": "none"},
        {"std": 0.05, "noise_model": "heteroscedastic"},
        {"std": 0.1, "noise_model": "heteroscedastic"},
        {"std": 0.2, "noise_model": "heteroscedastic"},
    ],
    synthesis_modes=[
        {"mode": mode, "strategies": ["baseline", "thinking", "vectorization", "guided"]}
        for mode in ["explicit", "implicit"]
    ],
    iterations=10,
    budget=100_000,
    timeout_seconds=30.0,
    runs_per_config=1,
    auto_resume=True,
    skip_completed=True,
    retry_failed_synthesis=True,
)
if matrix_df_result.is_success:
    matrix_df, summary = matrix_df_result.value
    display(matrix_df) if "display" in globals() else print(matrix_df.to_string())
else:
    matrix_df = None
    summary = None
    print(f"Error: {matrix_df_result.error.message}")



🎯 Synthesis Matrix Audit for 'gwdg.qwen3-coder-next':
                                                         Target Runs  Completed  Running Records  Failed Records      Status
Problem         Dimension Environment     Strategy                                                                          
f1 (Sphere)     2D        Explicit (0.0)  Baseline                 1          1                0               0  ✅ Complete
                                          Thinking                 1          1                0               0  ✅ Complete
                                          Vectorization            1          1                0               0  ✅ Complete
                                          Guided                   1          1                0               0  ✅ Complete
                          Implicit (0.0)  Baseline                 1          1                0               0  ✅ Complete
                                          Thinking                 1  

## 3. Parallel Multi-Process Experiment Execution (With Auto-Resumption & DB Sync)
Automatically queries `data/db.sqlite3` to check which experiments are still pending or incomplete:
1. **Interrupted Runs (`status == 'running'`)**: Resumed only where a champion is still missing, or to fulfill an explicit manual request. Extra queued records do not invalidate an existing champion.
2. **Completed Runs (`status == 'completed'`)**: A finite final champion makes the condition complete regardless of failures among its 10 candidate iterations.
3. **Failed Synthesis Sessions**: Retried automatically only when the condition has no completed valid champion and `retry_failed_synthesis = true`. Poor benchmark performance instead motivates explicit `rerun_experiment_ids`.
4. **Pending Runs**: In full-matrix mode, fresh tasks fill remaining protocol replicates. Recovery mode only resumes/retries eligible conditions and explicit manual requests.

All tasks execute concurrently across worker processes using `ProcessWorkerPool` with safe SQLite WAL concurrency.

In [25]:
# Recover the configured matrix, preserving completed experiments.
# #7253 has no checkpoint: replace its one saved iteration under the same ID.
# Remove rerun_experiment_ids after successful recovery to avoid repeating it.
results_result = campaign.run(
    problem_targets=[{"id": problem, "dimensions": [2, 10, 50]} for problem in [1, 8, 11, 15, 21]],
    noise_conditions=[
        {"std": 0.0, "noise_model": "none"},
        {"std": 0.05, "noise_model": "heteroscedastic"},
        {"std": 0.1, "noise_model": "heteroscedastic"},
        {"std": 0.2, "noise_model": "heteroscedastic"},
    ],
    synthesis_modes=[
        {"mode": mode, "strategies": ["baseline", "thinking", "vectorization", "guided"]}
        for mode in ["explicit", "implicit"]
    ],
    iterations=10,
    budget=100_000,
    timeout_seconds=30.0,
    runs_per_config=1,
    auto_resume=True,
    skip_completed=True,
    retry_failed_synthesis=True,
)
if results_result.is_success:
    results = results_result.value
else:
    results = None
    print(f"Error: {results_result.error.message}")


✅ No eligible tasks for this request and active model 'gwdg.qwen3-coder-next'. Nothing to run.


## Run one experiment without saving study data

Run the import cell, then run only the cell below. The service-connection, audit
and campaign cells are not needed for this test. Do not use Run All.

A separate `single_synthesis` service is created for this cell.
The transient provider uses in-memory experiment and code repositories and
isolated temporary checkpoints. It never opens the research database or writes
to the study code/checkpoint directories. Temporary checkpoints are removed when
the scope closes, including on failure. Operational logs and gateway throttle
state remain available normally.

This runs f1 / 2D / clean / Explicit Baseline for 10 generations on instance 1.
Each candidate receives 100,000 objective calls and a 30-second execution timeout;
the LLM API request timeout is separate. The result and generated champion code
remain in `single_result` after cleanup. Its experiment ID is local to memory,
not a database ID. This mode cannot resume after the process exits.

Setup explicitly enables one short model-generation probe before injecting the
client. A failed probe stops setup; the synthesis service does not manage
connections. Workers do not repeat this startup probe.

Use the normal provider and campaign cells for persistent experiments.


In [26]:
# from evolution.application.synthesis.service import Synthesis
# from evolution.infra.di import TransientEvolutionProvider

# transient_services = WorkflowScope(
#     TransientEvolutionProvider(test_generation_on_startup=True), context={}, request_context={}
# )
# try:
#     print("Checking model access and one short generation request...", flush=True)
#     single_synthesis = transient_services.get(Synthesis)
#     single_result = single_synthesis.run(
#         problem_id=1,
#         dim=2,
#         noise_std=0.0,
#         noise_model="none",
#         mode="explicit",
#         prompt_strategy="baseline",
#         budget=100_000,
#         iterations=10,
#         timeout_seconds=30.0,
#         synthesis_seed=43,
#         instance_id=1,
#     )
#     if single_result.is_success:
#         print(f"Transient synthesis: best error = {single_result.value.best_error}")
#     else:
#         print(single_result.error.message)
# finally:
#     transient_services.close()
